# 04 — Treatment-Effect Analysis

This notebook demonstrates the statistical-analysis and visualization modules. The implementation lives in `src/statistics.py` and `src/visualization.py`.

In [ ]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)

In [ ]:
import textwrap
import pandas as pd

from src.config import (
    METRICS,
    SCORED_OUTPUTS_FILENAME,
    get_data_dir,
    get_results_dirs,
)
from src.statistics import (
    make_paired_effects,
    summarize_treatment_effects,
    grouped_treatment_effects,
    build_qualitative_examples,
)
from src.visualization import (
    plot_average_treatment_effects,
    plot_group_effects,
)

DATA_DIR = get_data_dir(PROJECT_DIR)
RESULTS_DIR, TABLES_DIR, FIGURES_DIR = get_results_dirs(PROJECT_DIR)
SCORED_OUTPUTS_PATH = DATA_DIR / SCORED_OUTPUTS_FILENAME

pd.set_option("display.max_colwidth", 160)

print("Scored outputs path:", SCORED_OUTPUTS_PATH)
print("Tables directory:", TABLES_DIR)
print("Figures directory:", FIGURES_DIR)

## Load scored outputs

In [ ]:
scored_df = pd.read_csv(SCORED_OUTPUTS_PATH)

print(f"Loaded {len(scored_df):,} scored responses")
print(f"Unique prompts: {scored_df['prompt_id'].nunique():,}")
display(scored_df.head())

## Create paired prompt-level effects

Each row now represents one prompt, with base, instruct, and effect columns.

In [ ]:
paired = make_paired_effects(scored_df, metrics=METRICS)

paired_path = TABLES_DIR / "paired_prompt_effects.csv"
paired.to_csv(paired_path, index=False)

print(f"Paired analysis rows: {len(paired):,}")
print(f"Saved: {paired_path}")
display(paired.head())

## Overall treatment effects

In [ ]:
summary_df = summarize_treatment_effects(
    paired,
    metrics=METRICS,
    output_path=TABLES_DIR / "treatment_effects_summary.csv",
)

display(summary_df.round(4))

In [ ]:
plot_average_treatment_effects(
    summary_df,
    output_path=FIGURES_DIR / "average_treatment_effects.png",
)

## Domain-level treatment effects

In [ ]:
domain_effects = grouped_treatment_effects(
    paired,
    group_col="domain",
    metrics=METRICS,
    output_path=TABLES_DIR / "domain_treatment_effects.csv",
)

display(domain_effects)

In [ ]:
plot_group_effects(
    paired,
    group_col="domain",
    figures_dir=FIGURES_DIR,
    metrics=METRICS,
    prefix="domain",
)

## Prompt-type treatment effects

In [ ]:
prompt_type_effects = grouped_treatment_effects(
    paired,
    group_col="prompt_type",
    metrics=METRICS,
    output_path=TABLES_DIR / "prompt_type_treatment_effects.csv",
)

display(prompt_type_effects)

In [ ]:
plot_group_effects(
    paired,
    group_col="prompt_type",
    figures_dir=FIGURES_DIR,
    metrics=METRICS,
    prefix="prompt_type",
)

## Qualitative high-difference examples

In [ ]:
qualitative_df = build_qualitative_examples(
    paired,
    scored_df,
    output_path=TABLES_DIR / "qualitative_high_difference_examples.csv",
)

display(qualitative_df[[
    "metric", "prompt_id", "domain", "topic", "prompt_type",
    "base_score", "instruct_score", "effect"
]].head(20))

In [ ]:
def shorten(text, width=600):
    text = str(text).replace("\n", " ")
    return textwrap.shorten(text, width=width, placeholder=" ...")

for _, row in qualitative_df.head(5).iterrows():
    print("=" * 100)
    print(f"Metric: {row['metric']}")
    print(f"Prompt ID: {row['prompt_id']}")
    print(f"Domain/topic/type: {row['domain']} / {row['topic']} / {row['prompt_type']}")
    print(f"Effect: {row['effect']} | base={row['base_score']} instruct={row['instruct_score']}")
    print("Base response:")
    print(shorten(row["base_response"]))
    print("Instruction-tuned response:")
    print(shorten(row["instruct_response"]))
    print()

## Analysis complete

The resulting tables and figures are saved under `results/`.